# Inspect raw scouting data

Understat player-season stats and the Transfermarkt snapshot (players, valuations, clubs, transfers).

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
UNDERSTAT_PATH = RAW / "understat_player_season.csv"
TM = RAW / "transfermarkt"

players = pd.read_csv(TM / "players.csv")
valuations = pd.read_csv(TM / "player_valuations.csv")
clubs = pd.read_csv(TM / "clubs.csv")
transfers = pd.read_csv(TM / "transfers.csv")

print("players", players.shape)
print("player_valuations", valuations.shape)
print("clubs", clubs.shape)
print("transfers", transfers.shape)
print("players columns:", list(players.columns))

understat = None
if UNDERSTAT_PATH.exists():
    understat = pd.read_csv(UNDERSTAT_PATH)
    print("understat", understat.shape)
    print("understat columns:", list(understat.columns))
else:
    print(
        "understat CSV missing. collect_understat failed because "
        "https://understat.com is serving a maintenance page instead of JSON."
    )


players (50149, 26)
player_valuations (656301, 6)
clubs (796, 17)
transfers (175165, 10)
players columns: ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']
understat CSV missing. collect_understat failed because https://understat.com is serving a maintenance page instead of JSON.


## Understat: players per league and season

In [2]:
if understat is None:
    print("Skipped: no Understat file.")
else:
    counts = understat.groupby(["league", "season"], dropna=False).size().rename("players")
    display(counts.to_frame())


Skipped: no Understat file.


## Tottenham, Understat 2025-26

In [3]:
if understat is None:
    print("Skipped: no Understat file.")
else:
    season = understat["season"].astype(str)
    spurs_us = understat[
        understat["team"].astype(str).str.contains("Tottenham", case=False, na=False)
        & season.isin(["2526", "2025-26", "2025/26"])
    ]
    cols = ["player", "position", "minutes", "goals", "xg", "xa"]
    display(spurs_us[cols].sort_values("minutes", ascending=False))


Skipped: no Understat file.


## Tottenham, Transfermarkt (current squad)

In [4]:
spurs_tm = players[players["current_club_name"].eq("Tottenham Hotspur")].copy()
cols = ["name", "date_of_birth", "position", "market_value_in_eur"]
display(spurs_tm[cols].sort_values("market_value_in_eur", ascending=False))
print(f"{len(spurs_tm)} players currently at Tottenham Hotspur")


,name,date_of_birth,position,market_value_in_eur
39819,Luka Vuskovic,2007-02-24 00:00:00,Defender,60000000.0
29923,Micky van de Ven,2001-04-19 00:00:00,Defender,50000000.0
29459,Mohammed Kudus,2000-08-02 00:00:00,Attack,50000000.0
21799,Cristian Romero,1998-04-27 00:00:00,Defender,45000000.0
40661,Archie Gray,2006-03-12 00:00:00,Midfield,35000000.0
...,...,...,...,...
45352,Malachi Hardy,2008-03-10 00:00:00,Defender,NaN
45355,Luca Williams-Barnett,2008-10-01 00:00:00,Midfield,NaN
47138,Jun'ai Byfield,2008-12-06 00:00:00,Defender,NaN
47948,Tynan Thompson,2008-04-17 00:00:00,Attack,NaN


80 players currently at Tottenham Hotspur


## Name differences

In [5]:
if understat is None:
    print("Skipped: need both sources to compare names.")
else:
    season = understat["season"].astype(str)
    us_names = set(
        understat.loc[
            understat["team"].astype(str).str.contains("Tottenham", case=False, na=False)
            & season.isin(["2526", "2025-26", "2025/26"]),
            "player",
        ].astype(str)
    )
    tm_names = set(spurs_tm["name"].astype(str))
    only_us = sorted(us_names - tm_names)
    only_tm = sorted(tm_names - us_names)
    print(f"Understat names not in Transfermarkt ({len(only_us)}):")
    print("\n".join(only_us[:40]))
    print(f"\nTransfermarkt names not in Understat ({len(only_tm)}):")
    print("\n".join(only_tm[:40]))


Skipped: need both sources to compare names.


## Understat players with under 900 minutes

In [6]:
if understat is None:
    print("Skipped: no Understat file.")
else:
    low_minutes = understat.loc[understat["minutes"] < 900].groupby("season").size().rename("players_under_900")
    display(low_minutes.to_frame())


Skipped: no Understat file.
